# Layer-selective LoRA fine-tuning — Qwen3-VL-2B-Thinking on spatial reasoning

Fine-tune [`Qwen/Qwen3-VL-2B-Thinking`](https://huggingface.co/Qwen/Qwen3-VL-2B-Thinking) on the [SpaceThinker](https://huggingface.co/datasets/remyxai/SpaceThinker) + [SpaceOm](https://huggingface.co/datasets/remyxai/SpaceOm) datasets via TRL's `SFTTrainer`, restricting LoRA adapters to the **last 10 of 28 decoder layers** (L18–L27).

**Why layer-selective**: a [TransformerLens probe on the Space\* family](https://colab.research.google.com/drive/14p2l_RJKk5nqPHM3rkvCUKqGcse6XZ25?usp=sharing) found that spatial-reasoning capability crystallises in the last ~30% of decoder layers on Qwen2.5-VL-3B. Applying the same depth fraction to Qwen3-VL-2B's 28-layer decoder targets L18–L27. Skipping the early scene-understanding layers keeps the adapter surface small and preserves the base model's pretrained perception.

**Why the Thinking variant + Space datasets**: both datasets ship a `reasoning` field alongside `input`/`output`, structured for CoT supervision. The Thinking variant is pretrained to emit reasoning traces before the final answer — the pairing lets fine-tuning shape the *reasoning trajectory*, not just the terminal answer.

**Recipe**: LoRA r=256 / α=512 on `q,k,v,o,gate,up,down` (all-linear per block) × 10 late layers, lr 2e-4 cosine, warmup 3%, effective batch 128, 3 epochs, bf16, gradient checkpointing off, best-checkpoint selection on eval loss.

**Hardware**: A100 80GB (Colab Pro). Wall-clock ~70 min for the combined 12k-example train set at 3 epochs.


## Environment

In [ ]:
!nvidia-smi -L
import torch, sys
print(f'python {sys.version.split()[0]}, torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB')


In [ ]:
# Pillow 12 removed PIL._typing._Ink which qwen-vl-utils / transformers image utils
# still import; pin <12 to avoid the ImportError on `from transformers import Qwen3VL...`.
!pip install --quiet 'torchao>=0.16'
!pip install --quiet 'pillow<12'
!pip install --quiet -U \
    "transformers>=4.57" "trl>=0.14" "peft>=0.14" \
    "datasets>=2.20" "accelerate>=0.34" \
    qwen-vl-utils

# If this cell was run in a session that already imported PIL (with 12.x loaded into
# the kernel), Runtime → Restart runtime is required so the pinned 11.x actually
# takes effect. If the next cell still errors on _Ink, that's the fix.
import PIL, sys
print(f'pillow: {PIL.__version__}   (need <12)')


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets')
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()


## Load Qwen3-VL-2B-Thinking


In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = 'Qwen/Qwen3-VL-2B-Thinking'

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Reference config sanity-check
tc = model.config.text_config
print(f'model: {MODEL_ID}')
print(f'  n_layers: {tc.num_hidden_layers}')
print(f'  d_model:  {tc.hidden_size}')
print(f'  n_heads:  {tc.num_attention_heads}  (KV heads: {tc.num_key_value_heads}, GQA)')


## Layer-selective LoRA config

Adapt only the last 5 decoder layers (L23-L27 of 28). Matches the 82-96% depth range where our MI probe located the concept-to-answer computation on Qwen2.5-VL-3B.

Freeze the vision encoder explicitly — standard VLM PEFT practice, and consistent with our MI finding that spatial capability installs on the LM decoder.


In [ ]:
from peft import LoraConfig, get_peft_model

n_layers = model.config.text_config.num_hidden_layers
LATE_STACK_LAYERS = list(range(n_layers - 10, n_layers))  # last 10 layers = L18-L27 on 28-layer decoder
print(f'Adapting decoder layers {LATE_STACK_LAYERS[0]}-{LATE_STACK_LAYERS[-1]} of {n_layers} total')

# Freeze vision encoder BEFORE wrapping in PEFT.
# Qwen VLM versions nest the vision tower under different attributes
# (Qwen2.5-VL: model.visual; Qwen3-VL: model.model.visual). Use name-pattern
# matching so this works regardless of the exact attribute path.
n_frozen = 0
for name, p in model.named_parameters():
    if 'visual' in name.lower() or 'vision' in name.lower():
        p.requires_grad = False
        n_frozen += p.numel()
print(f'Frozen {n_frozen:,} vision-tower parameters via name-pattern match')

lora_config = LoraConfig(
    r=256,
    lora_alpha=512,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],  # all-linear
    layers_to_transform=LATE_STACK_LAYERS,
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Load SpaceThinker + SpaceOm (combined)


In [ ]:
from datasets import load_dataset, concatenate_datasets

# Combine SpaceThinker (11,413 train / 1,250 test, multi-turn ~5 pairs/example)
# with SpaceOm (729 / 81, single-turn). Combined: 12,142 train / 1,331 test.
# The multi-turn masking collator (below) unmasks every assistant span so both
# dataset shapes contribute assistant-token loss.
st = load_dataset('remyxai/SpaceThinker')
so = load_dataset('remyxai/SpaceOm')

ds = {
    'train': concatenate_datasets([st['train'], so['train']]).shuffle(seed=42),
    'test':  concatenate_datasets([st['test'],  so['test']]).shuffle(seed=42),
}
print(f'combined train: {len(ds["train"])}   (SpaceThinker {len(st["train"])} + SpaceOm {len(so["train"])})')
print(f'combined test:  {len(ds["test"])}    (SpaceThinker {len(st["test"])} + SpaceOm {len(so["test"])})')

print()
print('First train example (post-shuffle):')
ex = ds['train'][0]
print(f'  images: {len(ex["images"])} images, first: {ex["images"][0].size}')
print(f'  messages ({len(ex["messages"])} turns):')
for m in ex['messages'][:2]:
    role = m['role']
    content = m['content'][:1] if isinstance(m['content'], list) else str(m['content'])[:200]
    print(f'    [{role}] {content}')

# --- Chat-template sanity check — confirm assistant marker still lands correctly for both dataset origins ---
print()
print('=== Chat template rendering (first 900 chars) ===')
rendered = processor.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
print(rendered[:900])
print('...' if len(rendered) > 900 else '')
print()
marker = '<|im_start|>assistant\n'
n_markers = rendered.count(marker)
print(f'assistant marker occurrences in this example: {n_markers}  (SpaceThinker: ~5, SpaceOm: 1)')
if n_markers == 0:
    print('WARNING: assistant marker not found — collator will mask entire sequence!')


## Multimodal data collator

TRL's built-in dataset prep is text-only. For VLM SFT we set `remove_unused_columns=False` and `skip_prepare_dataset=True`, then supply a custom `data_collator` that applies the chat template and processor per batch.

Labels mask out padding, image tokens, and vision start/end markers so the loss is computed only on the answer text.


In [ ]:
def collate_fn(examples):
    """Multi-turn completion-only label masking.

    Initializes ALL labels to -100 (masked), then unmasks every
    <|im_start|>assistant\n ... <|im_end|> span so loss is computed on every
    assistant turn — not just the last. Critical for SpaceThinker's multi-turn
    examples (~5 assistant turns per example). Also correctly handles SpaceOm's
    single-turn examples (one unmask span).
    """
    import torch as _torch
    texts, images_list = [], []
    for ex in examples:
        text = processor.apply_chat_template(
            ex['messages'], tokenize=False, add_generation_prompt=False,
        )
        texts.append(text)
        images_list.append(ex['images'])

    batch = processor(text=texts, images=images_list, return_tensors='pt', padding=True)
    input_ids = batch['input_ids']

    # Default all labels to -100; we'll unmask assistant spans explicitly.
    labels = _torch.full_like(input_ids, -100)

    assist_marker_ids = processor.tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
    im_end_id = processor.tokenizer.convert_tokens_to_ids('<|im_end|>')
    marker_len = len(assist_marker_ids)

    for i in range(input_ids.size(0)):
        ids = input_ids[i].tolist()
        pos = 0
        while pos <= len(ids) - marker_len:
            if ids[pos:pos + marker_len] == assist_marker_ids:
                content_start = pos + marker_len
                content_end = content_start
                while content_end < len(ids) and ids[content_end] != im_end_id:
                    content_end += 1
                if content_end < len(ids):
                    # Unmask assistant content INCLUDING the <|im_end|> terminator
                    # so the model learns to close its turns.
                    labels[i, content_start:content_end + 1] = input_ids[i, content_start:content_end + 1]
                pos = content_end + 1
            else:
                pos += 1

    batch['labels'] = labels
    return batch


## SFTTrainer setup

Effective batch size = `per_device_train_batch_size * gradient_accumulation_steps = 32 * 4 = 128`. Gradient checkpointing is off (activations retained) — feasible on A100 80GB because the vision encoder and layers 0–17 are frozen.

Best-checkpoint selection on `eval_loss` keeps the run robust to late-training overfitting.


In [ ]:
from trl import SFTConfig, SFTTrainer

# Effective batch 128 (per_device=32, grad_accum=4) with GC off exploits A100 80GB's
# headroom (peak ~55 GB). ~285 steps at ~15s/step ≈ 70 min for 3 epochs on the combined set.
sft_config = SFTConfig(
    output_dir='./qwen3vl-thinking-combined-late-stack',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=32,
    gradient_checkpointing=False,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=True,
    tf32=True,
    report_to='none',
    remove_unused_columns=False,
    dataset_kwargs={'skip_prepare_dataset': True},
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    args=sft_config,
    data_collator=collate_fn,
    processing_class=processor,
)

trainer.train()


## Save adapter

In [ ]:
ADAPTER_NAME = 'qwen3vl-2b-thinking-space-lora'
trainer.save_model(f'./{ADAPTER_NAME}')

# Optional: push to Hub
# model.push_to_hub(f'remyxai/{ADAPTER_NAME}')
# processor.push_to_hub(f'remyxai/{ADAPTER_NAME}')
print(f'Saved to ./{ADAPTER_NAME}')


## Quick sanity-check generation on a held-out test example

In [ ]:
# Disable gradient checkpointing + enable KV cache for fast generation
model.gradient_checkpointing_disable()
model.config.use_cache = True

sample = ds['test'][0]

def _render_content(content):
    """SpaceOm messages have list-of-dict content; render the text parts as a string."""
    if isinstance(content, list):
        return ' '.join(c.get('text') or '<image>' for c in content).strip()
    return str(content)

user_msg = [m for m in sample['messages'] if m['role'] == 'user'][0]
gt_msg   = [m for m in sample['messages'] if m['role'] == 'assistant'][0]

prompt = processor.apply_chat_template([user_msg], tokenize=False, add_generation_prompt=True)
inputs = processor(text=[prompt], images=sample['images'], return_tensors='pt', padding=True).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)

generated = processor.batch_decode(
    out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
)[0]

print('=== Question ===')
print(_render_content(user_msg['content']))
print()
print('=== Model output ===')
print(generated)
print()
print('=== Ground truth ===')
print(_render_content(gt_msg['content']))


## Notes on the layer-selection choice

- **Transfer assumption**: L18–L27 are picked by depth-fraction from a TransformerLens probe on Qwen2.5-VL-3B (L30–L35 of 36 = 82–97% depth; L18–L27 of 28 = 64–96%). This assumes the *layer-wise* pattern transfers across Qwen VLM family versions. See the [Space\* mechanistic-inspection notebook](https://colab.research.google.com/drive/14p2l_RJKk5nqPHM3rkvCUKqGcse6XZ25?usp=sharing) for the probing tooling — running it against Qwen3-VL-2B directly would validate.
- **Baseline for comparison**: same recipe with `layers_to_transform` removed adapts all 28 layers with ~5.6× the trainable parameters. If eval accuracy is comparable to the layer-selective run, the depth-fraction transfer holds and layer selection is the parameter-efficiency win.
- **Rank rationale**: r=256 α=512 gives ~99M trainable (4.5% of a 2B model). For smaller datasets or aggressive parameter-efficiency, tighten to r=64/128 α=128/256.

## References

- **Base model**: [`Qwen/Qwen3-VL-2B-Thinking`](https://huggingface.co/Qwen/Qwen3-VL-2B-Thinking) — 28 layers, hidden 2048, 16 heads (GQA-8). CoT-tuned variant matching Space*'s `reasoning`-field annotations.
- **Datasets**: [`remyxai/SpaceThinker`](https://huggingface.co/datasets/remyxai/SpaceThinker), [`remyxai/SpaceOm`](https://huggingface.co/datasets/remyxai/SpaceOm)
- **VQASynth**: https://github.com/remyxai/VQASynth
- **TRL SFT for VLMs**: https://huggingface.co/docs/trl/sft_trainer
